# Motor thresholds — picked once, used everywhere

Every other notebook reads the thresholds from here. Pick a recording's thresholds in the cell
for it, press **Save**, and `functions.thresholds_for(recording)` returns them from then on —
in the per-session notebooks, in the comparisons, in the cross-participant figures.

**Why this notebook exists.** The same quantity used to be written three different ways: a single
`AMP_MT = 40` typed from the session log, a per-condition list typed into a config, and a
per-muscle dict from the picker. One threshold for the whole arm is wrong — muscles differ by
tens of mA — and a number typed from a log can't be checked against the trace.

**How it works.** A saved pick lives in `results/<participant-session>/mt_<recording>.csv`.
`thresholds_for()` returns it if it exists and falls back to automatic detection if it does not,
so nothing breaks while the picking is incomplete — `threshold_source()` tells you which you got.

**What to do in each cell.** Move the slider to the lowest trace carrying a real response. Green
dotted = where the detector put it, *Take detected* accepts it, *No response* leaves that muscle
out of every analysis. Then **Save**. Re-running a cell reloads what you saved, so you can correct.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from functions import (set_style, load_run, pretty, threshold_picker, save_threshold_csv,
                       load_threshold_csv, mt_file, thresholds_for, threshold_source)
from functions.paper import motor_thresholds
set_style()

## Settings, and what is already picked

In [ ]:
# ======== settings ========
EXCLUDE = ["Deltoid med.", "Biceps", "Triceps long"]   # never analysed: shown, but not picked
XLIM    = (-20, 130)                                   # time window of the stacked traces
CONSECUTIVE = 2        # intensities the automatic detector must see a response at
KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)
# ==========================

RECORDINGS = {}        # label -> csv, every train recording in the study
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
RECORDINGS["NTA · 30 Hz · cathodic · before"] = D + "Burst_autosave_20260724_102443_670ms.csv"
RECORDINGS["NTA · 30 Hz · cathodic · lidocaine"] = D + "Burst_autosave_20260724_113834_522ms.csv"
RECORDINGS["NTA · 30 Hz · anodic · before"] = D + "Burst_autosave_20260724_102721_694ms.csv"
RECORDINGS["NTA · 30 Hz · anodic · lidocaine"] = D + "Burst_autosave_20260724_114055_891ms.csv"
RECORDINGS["NTA · ARC-EX · cathodic · before"] = D + "Modulated_autosave_20260724_103530_508ms.csv"
RECORDINGS["NTA · ARC-EX · cathodic · lidocaine"] = D + "Modulated_autosave_20260724_114332_473ms.csv"
RECORDINGS["NTA · ARC-EX · anodic · before"] = D + "Modulated_autosave_20260724_103849_459ms.csv"
RECORDINGS["NTA · ARC-EX · anodic · lidocaine"] = D + "Modulated_autosave_20260724_114730_561ms.csv"
D = "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/"
RECORDINGS["P03 · 30 Hz · baseline"] = D + "Burst_autosave_20260715_164431_737ms.csv"
RECORDINGS["P03 · 30 Hz · vib applied, off"] = D + "Burst_autosave_20260715_165831_999ms.csv"
RECORDINGS["P03 · 30 Hz · vib ON extensors"] = D + "Burst_autosave_20260715_170629_569ms.csv"
RECORDINGS["P03 · 30 Hz · vib ON, 5 mA step"] = D + "Burst_autosave_20260715_172234_038ms.csv"
RECORDINGS["P03 · ARC-EX · baseline"] = D + "Modulated_autosave_20260715_164630_593ms.csv"
RECORDINGS["P03 · ARC-EX · vib applied, off"] = D + "Modulated_autosave_20260715_170032_049ms.csv"
RECORDINGS["P03 · ARC-EX · vib ON extensors"] = D + "Modulated_autosave_20260715_171433_605ms.csv"
D = "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/"
RECORDINGS["P04 · 30 Hz · baseline"] = D + "Burst_autosave_20260716_142311_232ms.csv"
RECORDINGS["P04 · 30 Hz · vib OFF"] = D + "Burst_autosave_20260716_144116_893ms.csv"
RECORDINGS["P04 · 30 Hz · vib ON flexors"] = D + "Burst_autosave_20260716_145100_665ms.csv"
RECORDINGS["P04 · 30 Hz · vib ON extensors"] = D + "Burst_autosave_20260716_152821_805ms.csv"
RECORDINGS["P04 · ARC-EX · baseline"] = D + "Modulated_autosave_20260716_142536_785ms.csv"
RECORDINGS["P04 · ARC-EX · vib OFF"] = D + "Modulated_autosave_20260716_144439_044ms.csv"
RECORDINGS["P04 · ARC-EX · vib ON flexors"] = D + "Modulated_autosave_20260716_145845_958ms.csv"
RECORDINGS["P04 · ARC-EX · vib ON extensors"] = D + "Modulated_autosave_20260716_153556_059ms.csv"
D = "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/"
RECORDINGS["P04 · 30 Hz · elec 1 · before"] = D + "Burst_autosave_20260717_150152_655ms.csv"
RECORDINGS["P04 · 30 Hz · elec 3 · before"] = D + "Burst_autosave_20260717_150503_708ms.csv"
RECORDINGS["P04 · 30 Hz · elec 1 · lidocaine"] = D + "Burst_autosave_20260717_161911_499ms.csv"
RECORDINGS["P04 · 30 Hz · elec 3 · lidocaine"] = D + "Burst_autosave_20260717_162022_520ms.csv"
RECORDINGS["P04 · ARC-EX · elec 1 · before"] = D + "Modulated_autosave_20260717_150657_684ms.csv"
RECORDINGS["P04 · ARC-EX · elec 3 · before"] = D + "Modulated_autosave_20260717_151033_569ms.csv"
RECORDINGS["P04 · ARC-EX · elec 1 · lidocaine"] = D + "Modulated_autosave_20260717_162347_270ms.csv"
RECORDINGS["P04 · ARC-EX · elec 3 · lidocaine"] = D + "Modulated_autosave_20260717_162548_293ms.csv"

def status():
    """Which recordings have hand-picked thresholds, and how many muscles each one keeps."""
    print(f"{'recording':44s}{'source':13s}  muscles kept")
    for lab, csv in RECORDINGS.items():
        src = threshold_source(csv)
        n = len(load_threshold_csv(mt_file(csv))) if src == "hand-picked" else None
        print(f"{lab:44s}{src:13s}  {n if n is not None else '-'}")
    done = sum(threshold_source(c) == "hand-picked" for c in RECORDINGS.values())
    print(f"\n{done} of {len(RECORDINGS)} recordings picked")

status()

In [ ]:
def pick(label):
    """Open the picker for one recording. Slider -> lowest trace with a real response, then Save."""
    csv = RECORDINGS[label]
    meta, t, sig = load_run(csv)
    chans = [c for c in sig if c != "Trigger A"
             and not any(pretty(c).startswith(x) for x in EXCLUDE)]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=CONSECUTIVE, verbose=False, **KW)["_"]
    print(f"{label}\n  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta]} mA"
          + ("  |  picks reloaded" if prev else ""))
    return threshold_picker(meta, t, sig, chans, xlim=XLIM, picks=prev, suggest=detected,
                            key=csv,
                            on_save=lambda p: save_threshold_csv(p, chans, csv, meta=meta))

## NTA · 24-07-2026 · polarity × lidocaine

In [ ]:
pick("NTA · 30 Hz · cathodic · before")

In [ ]:
pick("NTA · 30 Hz · cathodic · lidocaine")

In [ ]:
pick("NTA · 30 Hz · anodic · before")

In [ ]:
pick("NTA · 30 Hz · anodic · lidocaine")

In [ ]:
pick("NTA · ARC-EX · cathodic · before")

In [ ]:
pick("NTA · ARC-EX · cathodic · lidocaine")

In [ ]:
pick("NTA · ARC-EX · anodic · before")

In [ ]:
pick("NTA · ARC-EX · anodic · lidocaine")

## P03 · 15-07-2026 · tendon vibration

In [ ]:
pick("P03 · 30 Hz · baseline")

In [ ]:
pick("P03 · 30 Hz · vib applied, off")

In [ ]:
pick("P03 · 30 Hz · vib ON extensors")

In [ ]:
pick("P03 · 30 Hz · vib ON, 5 mA step")

In [ ]:
pick("P03 · ARC-EX · baseline")

In [ ]:
pick("P03 · ARC-EX · vib applied, off")

In [ ]:
pick("P03 · ARC-EX · vib ON extensors")

## P04 · 16-07-2026 · tendon vibration

In [ ]:
pick("P04 · 30 Hz · baseline")

In [ ]:
pick("P04 · 30 Hz · vib OFF")

In [ ]:
pick("P04 · 30 Hz · vib ON flexors")

In [ ]:
pick("P04 · 30 Hz · vib ON extensors")

In [ ]:
pick("P04 · ARC-EX · baseline")

In [ ]:
pick("P04 · ARC-EX · vib OFF")

In [ ]:
pick("P04 · ARC-EX · vib ON flexors")

In [ ]:
pick("P04 · ARC-EX · vib ON extensors")

## P04 · 17-07-2026 · lidocaine

In [ ]:
pick("P04 · 30 Hz · elec 1 · before")

In [ ]:
pick("P04 · 30 Hz · elec 3 · before")

In [ ]:
pick("P04 · 30 Hz · elec 1 · lidocaine")

In [ ]:
pick("P04 · 30 Hz · elec 3 · lidocaine")

In [ ]:
pick("P04 · ARC-EX · elec 1 · before")

In [ ]:
pick("P04 · ARC-EX · elec 3 · before")

In [ ]:
pick("P04 · ARC-EX · elec 1 · lidocaine")

In [ ]:
pick("P04 · ARC-EX · elec 3 · lidocaine")

## Progress

In [ ]:
status()